Welcome to my code for the Kaggle movie review competition.

The logic for this classifier is as follows:

First, identify whether the text is a movie reveiw or not.

Second, identify whether the movie reviews are positive or negative.

Because these two steps will likely benefit from different features, I decided to use two consecutive logicstic regression models.

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
np.random.seed(42)

In [52]:
# Load dependencies
from typing import Iterator, Iterable, List, Tuple, Text, Union
from sklearn.metrics import accuracy_score, precision_score, f1_score
import numpy as np
import pandas as pd
import re
import html

from scipy.sparse import spmatrix

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from langdetect import detect, LangDetectException

# Also note
NDArray = Union[np.ndarray, spmatrix]

ModuleNotFoundError: No module named 'langdetect'

In [49]:
# Read in the training data
df_all = pd.read_csv("train.csv")

## Clean the text

In [46]:
def clean_text(text: str) -> str:
    """
    Cleans raw text by removing HTML tags and.
    """
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [50]:
df_all['TEXT'] = df_all['TEXT'].apply(clean_text)

In [51]:
# Add in binary variables for the movie and valence
df_all['MOVIE_LABEL'] = df_all['LABEL'].replace({0: 0, 1: 1, 2: 1})
df_all['VAL_LABEL'] = df_all['LABEL'].replace({0: np.nan, 1: 0, 2: 1})
df_all[0:9]

,ID,TEXT,LABEL,MOVIE_LABEL,VAL_LABEL
0,327995652116863138,Carolyn Baker defines the niche of helping peo...,0,0,NaN
1,11848336500074516230,Just getting released from a six month drug re...,1,1,0.0
2,8453485550425672763,I was greatly dissappointed when I popped this...,0,0,NaN
3,13088897910749354342,This is a film that has garnered any interest ...,2,1,1.0
4,4199320520317837800,This is one of the more adorable episodes of t...,1,1,0.0
5,3178394667960127642,"Yes, the 13 year olds(and anyone older)who say...",2,1,1.0
6,4441347800848407828,Superbly researched. Cogent presentation of th...,0,0,NaN
7,3033277043686083675,"St. Teresa is a sad, dying church but the Card...",0,0,NaN
8,12332986805313571562,I don't remember if I finished this book. The ...,0,0,NaN


In [31]:
# See Frequency of labels in all data
print("LABEL: ", df_all['LABEL'].value_counts())
print("MOVIE LABEL: ", df_all['MOVIE_LABEL'].value_counts())
print("VAL LABEL: ", df_all['VAL_LABEL'].value_counts())

LABEL:  0    32289
1    19139
2    18889
Name: LABEL, dtype: int64
MOVIE LABEL:  1    38028
0    32289
Name: MOVIE_LABEL, dtype: int64
VAL LABEL:  0.0    19139
1.0    18889
Name: VAL_LABEL, dtype: int64


In [26]:
# Training file has 70,317 observations.
# Split into training and dev sets (assuming they are randomly ordered)
# dev set = 3%
df_train = df_all[0:68208]
df_dev = df_all[68208:]
print("Training Data Length: ",len(df_train))
print("Dev Data Length: ", len(df_dev))

Training Data Length:  68208
Dev Data Length:  2109


In [30]:
# See Frequency of labels in training vs dev
print("Training\nLABEL:\n", df_train['LABEL'].value_counts())
print("MOVIE LABEL:\n", df_train['MOVIE_LABEL'].value_counts())
print("VAL LABEL:\n", df_train['VAL_LABEL'].value_counts())
print("\n")
print("Dev\nLABEL:\n", df_dev['LABEL'].value_counts())
print("MOVIE LABEL:\n", df_dev['MOVIE_LABEL'].value_counts())
print("VAL LABEL:\n", df_dev['VAL_LABEL'].value_counts())

# All seems equal enough

Training
LABEL:
 0    31343
1    18564
2    18301
Name: LABEL, dtype: int64
MOVIE LABEL:
 1    36865
0    31343
Name: MOVIE_LABEL, dtype: int64
VAL LABEL:
 0.0    18564
1.0    18301
Name: VAL_LABEL, dtype: int64


Dev
LABEL:
 0    946
2    588
1    575
Name: LABEL, dtype: int64
MOVIE LABEL:
 1    1163
0     946
Name: MOVIE_LABEL, dtype: int64
VAL LABEL:
 1.0    588
0.0    575
Name: VAL_LABEL, dtype: int64


# Identifying movie reviews

This next section will be devoted to creating a classifier that can identify if the given post is a movie review or not.

In [38]:
class TextToFeatures:
    def __init__(self):
        """
        Initializes an object for converting texts to features.    
        """
    self.vectorizer = CountVectorizer(analyzer='char_wb', ngram_range=(3, 5))
    # Character n-gram used to reduce noice from missing space typos.
    
    def fit(self, training_texts: Iterable[Text]) -> None:
        """
        Fits ("trains") a TextToFeature instance on a collection of documents.
        
        The provided training texts are analyzed to determine the vocabulary, 
        i.e., all feature values that the converter will support. 
        Each such feature value will be associated with a unique integer index 
        that may later be accessed via the .index() method.

        It is up to the implementer exactly what features to produce from a
        text, but the features will always include some single words and some
        multi-word expressions (e.g., "need" and "to you").
        
        t2f = TextToFeatures()
        t2f.fit(docs)

        :param training_texts: The training texts.
        """
        self.vectorizer.fit(training_texts)
        
        
    def index(self, feature: Text) -> Union[None, int]:
        """
        Returns the index in the vocabulary of the given feature value.  
        If the features isn't present, return None.

        :param feature: A feature
        :return: The unique integer index associated with the feature or None if not present.
        """
        return self.vectorizer.vocabulary_.get(feature) 
        # Note to self: .get() returns None if the thing is not present

    def transform(self, texts: Iterable[Text]) -> NDArray:
        """
        Creates a feature matrix from a sequence of texts.
        
        
        t2f = TextToFeatures()
        t2f.fit(docs)

        # this produces a NDArray representing our features for the provided doc
        t2f.transform(["Let's meet at Coyote Joe's at 6."])

        Each row of the matrix corresponds to one of the input texts. The value
        at index j of row i is the value in the ith text of the feature
        associated with the unique integer j.

        It is up to the implementer what the value of a feature that is present
        in a text should be, though a common choice is 1. Features that are
        absent from a text will have the value 0.

        :param texts: A sequence of texts.
        :return: A matrix, with one row of feature values for each text.
        """
        return self.vectorizer.transform(texts)

In [39]:
class TextToLabels:
    def __init__(self):
        """
        Initializes an object for converting texts to labels.
        """
        self.encoder = LabelEncoder()

    def fit(self, training_labels: Iterable[Text]) -> None:
        """
        Assigns each distinct label a unique integer.
        
        
        Training labels are analyzed to determine the vocabulary, 
        i.e., all labels that the converter will support. 
        Each such label will be associated with a unique integer index 
        that may later be accessed via the .index() method.

        :param training_labels: The training labels.
        """
        self.encoder.fit(training_labels)
        
    def index(self, label: Text) -> Union[None, int]:
        """Returns the index in the vocabulary of the given label.

        :param label: A label
        :return: The unique integer index associated with the label.
        """
        matches = np.where(self.encoder.classes_ == label)[0] # [0] is bc np.where gives a tuple, and we only want the first part
        # Need to account for matches being empty
        if matches.size == 0:
            return None
        return int(matches[0])

    def transform(self, labels: Iterable[Text]) -> NDArray:
        """
        Creates a label vector from a sequence of labels.

        Each entry in the vector corresponds to one of the input labels. The
        value at index j is the unique integer associated with the jth label.

        :param labels: A sequence of labels.
        :return: A vector, with one entry for each label.
        """
        return self.encoder.transform(labels) 
        
    def __contains__(self, label: Text) -> bool:
        """
        Special "dunder" method to check if a label is known to the TextToLabels instance.
        
        labeler = TextToLabels()
        labeler.fit(["POSITIVE", "NEGATIVE"])

        # should be True:
        "POSITIVE" in labeler 
        
        # should be False:
        "MBOP" in labeler
        
        :return: True if the label was seen in the training data; False otherwise
        """
        return False if self.index(label) is None else True

In [40]:
class Classifier:
    def __init__(self):
        """
        Initalizes a logistic regression classifier.
        """
        self.classifier = LogisticRegression(max_iter=1000)

        
    def train(self, features: NDArray, labels: NDArray) -> None:
        """
        Trains the classifier using the given training examples.

        :param features: A feature matrix, where each row represents a text.
        Such matrices will typically be generated via TextToFeatures.
        :param labels: A label vector, where each entry represents a label.
        Such vectors will typically be generated via TextToLabels.
        """
        self.classifier.fit(features, labels)
    
    # just an alias for "train"
    fit = train
    
    def predict(self, features: NDArray) -> NDArray:
        """Makes predictions for each of the given examples.

        :param features: A feature matrix, where each row represents a text.
        Such matrices will typically be generated via TextToFeatures.
        :return: A prediction vector, where each entry represents a label.
        """
        return self.classifier.predict(features)

## Testing the movie classifier

In [41]:
# Training on movie labels
t2f = TextToFeatures()
t2l = TextToLabels()
clf = Classifier()

# Process the text
t2f.fit(df_train['TEXT'])
train_features = t2f.transform(df_train['TEXT'])

# Process the labels
t2l.fit(df_train['MOVIE_LABEL'])
train_labels = t2l.transform(df_train['MOVIE_LABEL'])

# Train the classifier
clf.train(train_features, train_labels)


ValueError: np.nan is an invalid document, expected byte or unicode string.

In [43]:
# Testing movie classification on dev set

# Text to features
dev_features = t2f.transform(df_dev['TEXT'])

# Text to labels
dev_true_labels = t2l.transform(df_dev['MOVIE_LABEL'])

# Generate predictions
dev_predictions = clf.predict(dev_features)

# Combine output with the development dataframe
# We use .inverse_transform on the LabelEncoder to turn the integer 
# predictions back into readable text (e.g., 0 -> "NEGATIVE")
df_dev['MOVIE_PRED_LABEL'] = dev_predictions

# View a sample of the combined dataframe
print(df_dev.head())

NotFittedError: Vocabulary not fitted or provided

In [ ]:
# Movie classifier metrics
accuracy = accuracy_score(dev_true_labels, dev_predictions)
precision = precision_score(dev_true_labels, dev_predictions, average='weighted', zero_division=0)
f1 = f1_score(dev_true_labels, dev_predictions, average='weighted', zero_division=0)

# Print the final metrics
print("--- Movie Review Evaluation Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score:  {f1:.4f}")

## The Valence Classifier

Now that I can identify whether a post is about a movie, I will then identify whether it is positive or negative.

To train this classifier, I will filter out any cases that are not about movies. In testing, this will be based on the true labels, but in real use, it will have to be based on the output of the first classifier. 
This is probably a source of error.. but let's hope it's not too great.